# Sleep Data — Exploratory Analysis

3 years of sleep tracking (Feb 2023 – Mar 2026), extracted from the Napper app.

**Key life events for context:**
- **Feb 2023**: Started tracking (~3 months old)
- **Q2 2023**: 3 naps/day, nursing + bottles
- **Q4 2023–Q1 2024**: Nap transition (3→2→1)
- **Mid 2024 – Feb 2025**: Family discord period
- **~Feb 2025**: Leave of absence begins
- **Late Jun 2025**: Moved from Mountain Time to Wisconsin (Central Time)
- **Jul–Aug 2025**: Living at grandparents' house
- **Sep 2025**: Moved into new house as family unit
- **~Oct 2025**: Back to work

All times normalized to **Central Time** (`_ct` columns) for consistent comparison.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

# Load data
conn = sqlite3.connect("sleep.db")
daily = pd.read_sql("SELECT * FROM napper_daily", conn)
events = pd.read_sql("SELECT * FROM napper_events", conn)
conn.close()

# Parse dates
daily["date"] = pd.to_datetime(daily["date"])

# Convert CT times to fractional hours for plotting
def time_to_hours(t):
    if pd.isna(t) or t is None:
        return None
    try:
        h, m = int(t[:2]), int(t[3:5])
        return h + m / 60
    except:
        return None

daily["wake_hr"] = daily["wake_time_ct"].apply(time_to_hours)
daily["bed_hr"] = daily["bedtime_ct"].apply(time_to_hours)
# Wrap post-midnight bedtimes for plotting
daily["bed_hr_plot"] = daily["bed_hr"].apply(lambda x: x + 24 if x is not None and x < 6 else x)
daily["nap_start_hr"] = daily["nap_start_ct"].apply(time_to_hours)
daily["nap_end_hr"] = daily["nap_end_ct"].apply(time_to_hours)

print(f"Loaded {len(daily)} days, {len(events)} events")
print(f"Date range: {daily['date'].min().date()} to {daily['date'].max().date()}")
daily.head()

## Life events overlay

Shaded regions mark key periods. Used on all charts below for context.

In [ ]:
# Life event annotations — reused across charts
EVENTS = [
    ("2024-06-01", "2025-02-28", "Family discord", "#ff9999", 0.12),
    ("2025-02-01", "2025-06-30", "Leave of absence", "#ffcc99", 0.12),
    ("2025-07-01", "2025-08-31", "At grandparents'", "#99ccff", 0.15),
    ("2025-09-01", "2025-09-30", "Moved in together", "#99ff99", 0.15),
]

EVENT_LINES = [
    ("2025-06-28", "TZ: Mountain → Central", "gray"),
    ("2025-10-01", "Back to work", "green"),
]

def add_life_events(ax):
    """Add shaded regions and event lines to any chart."""
    for start, end, label, color, alpha in EVENTS:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), 
                   alpha=alpha, color=color, label=label, zorder=0)
    for date, label, color in EVENT_LINES:
        ax.axvline(pd.Timestamp(date), color=color, linestyle="--", alpha=0.5, zorder=1)
        ax.text(pd.Timestamp(date), ax.get_ylim()[1], f" {label}", 
                fontsize=8, color=color, va="top", ha="left")

def format_time_axis(ax):
    """Format x-axis as dates, y-axis as HH:MM."""
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    def hour_fmt(x, _):
        h = int(x) % 24
        m = int((x % 1) * 60)
        return f"{h:02d}:{m:02d}"
    ax.yaxis.set_major_formatter(plt.FuncFormatter(hour_fmt))

print("Helper functions ready.")

## 1. Daily schedule — wake, nap, bedtime over 3 years

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# 30-day rolling averages
roll = daily.set_index("date")[["wake_hr", "nap_start_hr", "nap_end_hr", "bed_hr_plot"]].rolling("30D").mean()

# Plot raw data as faint dots
ax.scatter(daily["date"], daily["wake_hr"], s=4, alpha=0.15, color="orange", zorder=2)
ax.scatter(daily["date"], daily["bed_hr_plot"], s=4, alpha=0.15, color="navy", zorder=2)
ax.scatter(daily["date"], daily["nap_start_hr"], s=4, alpha=0.1, color="green", zorder=2)
ax.scatter(daily["date"], daily["nap_end_hr"], s=4, alpha=0.1, color="green", zorder=2)

# Nap band (shaded between nap start and end)
ax.fill_between(roll.index, roll["nap_start_hr"], roll["nap_end_hr"], 
                alpha=0.3, color="green", label="Nap window (30d avg)", zorder=3)

# Rolling average lines
ax.plot(roll.index, roll["wake_hr"], color="orange", linewidth=2, label="Wake time (30d avg)", zorder=4)
ax.plot(roll.index, roll["bed_hr_plot"], color="navy", linewidth=2, label="Bedtime (30d avg)", zorder=4)

add_life_events(ax)
format_time_axis(ax)

ax.set_ylim(5, 26)
ax.set_ylabel("Time of day (CT)")
ax.set_title("Daily Schedule — 3 Years of Sleep Data", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Bedtime consistency — variability as a stress indicator

In [ ]:
# Monthly bedtime standard deviation
monthly = daily.set_index("date").resample("M")["bed_hr_plot"].agg(["mean", "std", "count"])
monthly = monthly[monthly["count"] >= 5]

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(monthly.index, monthly["std"] * 60, width=25, color="steelblue", alpha=0.7, zorder=3)
ax.axhline(y=45, color="red", linestyle=":", alpha=0.5, label="45 min threshold")

add_life_events(ax)

ax.set_ylabel("Bedtime variability (std dev, minutes)")
ax.set_title("Bedtime Consistency by Month — Higher = More Chaotic", fontsize=14, fontweight="bold")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Nap evolution — duration and skip rate

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# --- Nap duration ---
has_nap = daily[daily["nap_duration_min"].notna()].copy()
roll_nap = has_nap.set_index("date")["nap_duration_min"].rolling("30D").mean()

ax1.scatter(has_nap["date"], has_nap["nap_duration_min"], s=6, alpha=0.2, color="green", zorder=2)
ax1.plot(roll_nap.index, roll_nap.values, color="darkgreen", linewidth=2, label="30-day avg", zorder=4)
add_life_events(ax1)
ax1.set_ylabel("Nap duration (minutes)")
ax1.set_title("Nap Duration Over Time", fontsize=13, fontweight="bold")
ax1.legend(fontsize=9)

# --- Monthly nap skip rate ---
monthly_nap = daily.set_index("date").resample("M").agg(
    total=("nap_skipped", "count"),
    skipped=("nap_skipped", "sum"),
    has_nap=("nap_duration_min", lambda x: x.notna().sum()),
)
# Days with no nap logged AND not marked skipped = also a skip
monthly_nap["no_nap_days"] = monthly_nap["total"] - monthly_nap["has_nap"]
monthly_nap["skip_rate"] = (monthly_nap["no_nap_days"] / monthly_nap["total"] * 100)
monthly_nap = monthly_nap[monthly_nap["total"] >= 5]

ax2.bar(monthly_nap.index, monthly_nap["skip_rate"], width=25, color="coral", alpha=0.7, zorder=3)
add_life_events(ax2)
ax2.set_ylabel("Nap skip rate (%)")
ax2.set_title("Monthly Nap Skip Rate", fontsize=13, fontweight="bold")
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

plt.tight_layout()
plt.show()

## 4. Total sleep window (bedtime to wake) and awake hours

In [ ]:
# Calculate overnight sleep window: bedtime → next day's wake
daily_sorted = daily.sort_values("date").copy()
daily_sorted["next_wake_hr"] = daily_sorted["wake_hr"].shift(-1)

# Sleep window = time from bed to next wake (crossing midnight)
# bed_hr_plot is already 24+ for post-midnight
daily_sorted["sleep_window_hr"] = daily_sorted["next_wake_hr"] + 24 - daily_sorted["bed_hr_plot"]
# Filter out unreasonable values
daily_sorted.loc[daily_sorted["sleep_window_hr"] > 16, "sleep_window_hr"] = None
daily_sorted.loc[daily_sorted["sleep_window_hr"] < 4, "sleep_window_hr"] = None

fig, ax = plt.subplots(figsize=(16, 5))

roll_sleep = daily_sorted.set_index("date")["sleep_window_hr"].rolling("30D").mean()
ax.scatter(daily_sorted["date"], daily_sorted["sleep_window_hr"], s=4, alpha=0.15, color="purple", zorder=2)
ax.plot(roll_sleep.index, roll_sleep.values, color="purple", linewidth=2, label="30-day avg", zorder=4)

add_life_events(ax)
ax.set_ylabel("Hours")
ax.set_title("Overnight Sleep Window (Bed → Wake)", fontsize=14, fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
plt.tight_layout()
plt.show()

## 5. Day-of-week patterns

In [ ]:
daily["dow"] = daily["date"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Wake time by day of week
sns.boxplot(data=daily, x="dow", y="wake_hr", order=dow_order, ax=ax1, 
            palette="YlOrRd", fliersize=2)
ax1.set_ylabel("Wake time (CT)")
ax1.set_title("Wake Time by Day of Week")
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=45)
# Format y-axis as time
def hr_fmt(x, _):
    return f"{int(x):02d}:{int((x%1)*60):02d}"
ax1.yaxis.set_major_formatter(plt.FuncFormatter(hr_fmt))

# Bedtime by day of week
sns.boxplot(data=daily, x="dow", y="bed_hr_plot", order=dow_order, ax=ax2,
            palette="PuBu", fliersize=2)
ax2.set_ylabel("Bedtime (CT)")
ax2.set_title("Bedtime by Day of Week")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=45)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(hr_fmt))

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path
from datetime import date

DATA_DIR = Path("processed")
DOB = date(2022, 11, 1)  # Approximate — adjust to your child's DOB

# ── Load & merge ──────────────────────────────────────────────────────────
daily = pd.read_csv(DATA_DIR / "napper_daily.csv", parse_dates=["date"])
events = pd.read_csv(DATA_DIR / "napper_events.csv", parse_dates=["date"])

daily["nap_skipped"] = daily["nap_skipped"].astype(str).str.lower() == "true"
events["skipped"] = events["skipped"].astype(str).str.lower() == "true"

# Aggregate nap events per day (handles multi-nap days in early data)
nap_ev = (
    events[(events["event_type"] == "NAP") & ~events["skipped"]
           & events["duration_min"].notna() & (events["duration_min"] > 0)]
    .drop_duplicates(subset=["date", "start_time_ct", "end_time_ct", "duration_min"])
)
nap_agg = nap_ev.groupby("date").agg(
    ev_nap_min=("duration_min", "sum"),
    nap_count=("duration_min", "size"),
).reset_index()

df = daily.merge(nap_agg, on="date", how="left").sort_values("date").reset_index(drop=True)

# ── Parse _ct (Central Time) columns ─────────────────────────────────────
def to_minutes(t):
    if pd.isna(t) or t == "": return np.nan
    parts = str(t).strip().split(":")
    return int(parts[0]) * 60 + int(parts[1])

def fmt12(m):
    if pd.isna(m): return ""
    h, mi = int(m // 60) % 24, int(m % 60)
    return f"{h % 12 or 12}:{mi:02d} {'AM' if h < 12 else 'PM'}"

for col, out in [("bedtime_ct", "bed"), ("wake_time_ct", "wake"),
                 ("nap_start_ct", "nap_start"), ("nap_end_ct", "nap_end")]:
    df[out] = df[col].apply(to_minutes)

df["total_nap"] = df["ev_nap_min"].fillna(df["nap_duration_min"])

# Nap count (0 for skipped days)
df.loc[df["nap_skipped"], "nap_count"] = 0
fallback = pd.Series(np.where(df["nap_duration_min"].notna() & (df["nap_duration_min"] > 0), 1, np.nan), index=df.index)
df["nap_count"] = df["nap_count"].fillna(fallback)

# ── Nighttime sleep: bed[N] → wake[N+1] ─────────────────────────────────
df["wake_next"] = df["wake"].shift(-1)
consecutive = (df["date"].shift(-1) - df["date"]).dt.days == 1
post_midnight = df["bed"] < 360  # CT conversion can push 23:XX past midnight

df["night"] = np.where(
    df["bed"].notna() & df["wake_next"].notna() & consecutive,
    np.where(post_midnight,
             df["wake_next"] - df["bed"],
             (1440 - df["bed"]) + df["wake_next"]),
    np.nan,
)
df.loc[(df["night"] < 180) | (df["night"] > 900), "night"] = np.nan  # 3–15h sanity

# ── Derived ──────────────────────────────────────────────────────────────
df["total_sleep"] = df["night"] + df["total_nap"].fillna(0)  # NaN if night is NaN
df["wake_window"] = df["bed"] - df["nap_end"]
df.loc[(df["wake_window"] < 0) | (df["wake_window"] > 720), "wake_window"] = np.nan

df["age_months"] = (df["date"] - pd.Timestamp(DOB)).dt.days / 30.44
brackets = [(0,12,"0-12mo"),(12,18,"12-18mo"),(18,24,"18-24mo"),
            (24,30,"2-2.5yr"),(30,36,"2.5-3yr"),(36,999,"3yr+")]
df["age"] = df["age_months"].apply(lambda m: next(l for lo,hi,l in brackets if lo <= m < hi))

df["dow"] = df["date"].dt.day_name()

# Convert to hours for readability
for col in ["night", "total_nap", "total_sleep", "wake_window"]:
    df[f"{col}_h"] = df[col] / 60

print(f"{len(df)} days | {df['night'].notna().sum()} with nighttime sleep | {df['date'].min().date()} to {df['date'].max().date()}")
df[["date", "age", "wake", "nap_end", "bed", "total_nap", "night", "total_sleep", "wake_window", "nap_count"]].tail(10)

## Notes from Streamlit exploration

**What the data shows (and doesn't):**

- **Nap duration does NOT predict bedtime** within any age bracket (all p > 0.25). The scatter plot looks like a relationship, but it's just age-driven clustering — short naps in infancy vs long naps now.
- **Nap duration DOES predict nighttime sleep** — every 15 min more nap → ~4.3 min less nighttime sleep (p < 0.0001). Small per-day effect but it compounds.
- **Short vs long nap days tell the story:** ≤60 min nap → 10.6h night, >90 min nap → 9.6h night. Full hour difference.
- **Wake window averages 6.3h** (recommended for age 3 is 5.5–6h). Running slightly long.
- **The dual-axis rolling average chart is the most compelling view** — shows clear inverse periods where nap and nighttime sleep trade off.
- **Day-of-week patterns**: Worth checking if Thu/Fri (Cory's days) differ from the rest.

**Data gotchas:**
- `napper_daily.csv` only stores ONE nap per day (the last one). For multi-nap periods (pre-2024), total nap time must come from the events table.
- Nighttime sleep = `bed[day N]` → `wake[day N+1]`. Only works for consecutive days.
- The CT conversion can push 23:XX bedtimes past midnight (e.g., 23:20 MST → 00:20 CT). The pipeline handles this — bedtimes before 6 AM are treated as post-midnight.
- `nap_duration_min` is unchanged by TZ conversion (durations are TZ-agnostic). Only clock times needed conversion.

## Nap Duration + Nighttime Sleep Over Time
The most compelling chart — 7-day rolling averages with inverse periods highlighted.

In [ ]:
c = df[["date", "total_nap", "night"]].copy().sort_values("date")
c["nap_7d"] = c["total_nap"].rolling(7, min_periods=3).mean()
c["night_7d"] = c["night"].rolling(7, min_periods=3).mean()
c = c.dropna(subset=["nap_7d", "night_7d"])

# Rolling 30-day correlation to find inverse periods
corr_df = df[["date", "total_nap", "night"]].copy().sort_values("date")
corr_df["rcorr"] = corr_df["total_nap"].rolling(30, min_periods=15).corr(corr_df["night"])

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=c["date"], y=c["nap_7d"], name="Nap (7d avg)",
    line=dict(color="#9b59b6", width=2),
    hovertemplate="%{x|%b %d, %Y}<br>Nap: %{y:.0f} min<extra></extra>",
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=c["date"], y=c["night_7d"], name="Night (7d avg)",
    line=dict(color="#3498db", width=2),
    hovertemplate="%{x|%b %d, %Y}<br>Night: %{y:.0f} min<extra></extra>",
), secondary_y=True)

# Shade inverse periods (rolling corr < -0.3)
inv = corr_df[corr_df["rcorr"] < -0.3]
if len(inv) > 0:
    inv_dates = inv["date"].sort_values().reset_index(drop=True)
    spans, start, prev = [], inv_dates.iloc[0], inv_dates.iloc[0]
    for d in inv_dates.iloc[1:]:
        if (d - prev).days > 3:
            spans.append((start, prev))
            start = d
        prev = d
    spans.append((start, prev))
    for s, e in spans:
        fig.add_vrect(x0=s, x1=e + pd.Timedelta(days=1),
                      fillcolor="rgba(231,76,60,0.12)", layer="below", line_width=0)

# Life event annotations
events_map = {
    "2024-10-01": "Family discord",
    "2025-02-01": "Leave of absence",
    "2025-07-01": "Move to WI",
    "2025-09-01": "New house",
    "2025-10-01": "Back to work",
}
for dt, label in events_map.items():
    fig.add_vline(x=dt, line_dash="dot", line_color="rgba(255,255,255,0.25)",
                  annotation_text=label, annotation_position="top",
                  annotation_font_color="rgba(255,255,255,0.5)", annotation_font_size=10)

fig.update_layout(
    template="plotly_dark", paper_bgcolor="#0e1117", plot_bgcolor="#1a1a2e",
    font=dict(color="#e0e0e0"), width=1000, height=450,
    yaxis=dict(title="Nap Duration (min)"),
    yaxis2=dict(title="Nighttime Sleep (min)"),
    legend=dict(x=0.01, y=0.99),
    margin=dict(t=30, b=40),
)
fig.show()